# MM5 lab: build a reliable link (30 min)

**What you get:** a simulator of one link from Alice to Bob. Frames are real bits; the channel adds delay and flips bits (the Binary Symmetric Channel from Lecture 5).
**What you build:** stop-and-wait ARQ, in `student/mm5_link.py`, with Copilot's help.
**What you find out:** things the paper exercises cannot reach, because they need thousands of frames.

| Part | Time | |
|---|---|---|
| A | 5 min | Why a link needs a protocol |
| B | 10 min | Implement stop-and-wait ARQ with Copilot |
| C | 13 min | Pick **two** experiments |
| D | 2 min | Wrap up |

**Rules.** Write your prediction *before* you run a cell. Copilot may write code in `student/`, never in `commsim/`. Answers go in the answer cells, in your own words.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")   # the CommSim folder
from commsim import *
from commsim.checks import check_mm5
from student.mm5_link import UtopiaSender, UtopiaReceiver, StopAndWaitSender, StopAndWaitReceiver

print(FrameFormat(max_payload_bytes=64, check="crc8").describe())

Compare this header with Figure 1 of Problem Set 1. The simulator adds one bit: the frame type (DATA or ACK).

## Part A: why a link needs a protocol (5 min)

Same setup as Problem Set 1, Problems 1 and 2: Alice sends 79 frames of 64 bytes at 1 kbit/s, and the one-way delay is 10 ms. The given **Utopia** protocol sends all frames back to back, and Bob accepts everything he receives.

In [ ]:
point_to_point(UtopiaSender, UtopiaReceiver, ber=0)

**A1.** Compare the goodput with your answer to Problem 1(c). Why is it not exactly the same?

> **Your answer:** *(write here)*

**A2. Predict first:** with BER = 10⁻³, how many of the 79 frames reach Bob intact? (Use the PER formula from Lecture 5.)

> **Your answer:** *(write here)*

In [ ]:
r = point_to_point(UtopiaSender, UtopiaReceiver, ber=1e-3)
print(r)
sequence_diagram(r, max_frames=6);

**A3.** Utopia fails in two different ways. Name them, and name the mechanism from Lecture 5 that fixes each one.

> **Your answer:** *(write here)*

## Part B: implement stop-and-wait ARQ (10 min)

1. Open `student/mm5_link.py`. Read the Utopia example and the specification above `StopAndWaitSender`.
2. Ask Copilot to implement `StopAndWaitSender` and `StopAndWaitReceiver` from the specification. Read what it wrote: you should be able to explain every line.
3. Run the check below. Fix and re-run until all checks pass. (The notebook reloads your file automatically.)

*Tip:* implement it **without** the sequence number first and run the check. Which check fails, and why?

In [ ]:
check_mm5(StopAndWaitSender, StopAndWaitReceiver)

In [ ]:
r = point_to_point(StopAndWaitSender, StopAndWaitReceiver, ber=1e-3)
print(r)
sequence_diagram(r, max_frames=10);

**B1.** Compare with Utopia at the same BER: what did ARQ buy, and what did it cost? In the diagram, find a retransmission. What triggered it?

> **Your answer:** *(write here)*

## Part C: pick two experiments (13 min)

Each experiment runs hundreds or thousands of frames. **Write your prediction first.**

### C1. What is the best frame size?
Problem 1(d) suggested that bigger frames are more efficient, because the overhead is paid once per frame. But a bigger frame is also more likely to be hit by a bit error. Link: 1 Mbit/s, 10 µs delay, CRC-16.

**Predict:** at BER = 10⁻⁴ and at BER = 10⁻³, roughly which payload size gives the highest goodput?

> **Your answer:** *(write here)*

In [ ]:
sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
curves = {}
for ber in [1e-4, 1e-3]:
    curves[f"BER = {ber:g}"] = [
        point_to_point(StopAndWaitSender, StopAndWaitReceiver, n_packets=300, payload_bytes=L,
                       rate_bps=1e6, prop_delay=1e-5, ber=ber, check="crc16").goodput_bps / 1e3
        for L in sizes]
sweep_plot(sizes, curves, "payload per frame (bytes)", "goodput (kbit/s)", logx=True,
           title="Best payload size for stop-and-wait");

**C1.** Where is the optimum for each BER? Explain the shape of the curve with the overhead and the PER formula.

> **Your answer:** *(write here)*

### C2. How far can stop-and-wait reach?
Link: 1 Mbit/s, 64-byte payloads, no bit errors. Signals travel at about 2 × 10⁸ m/s in cable. The pre-lecture reading mentioned Mars: 4 to 24 **minutes** one way.

**Predict:** at what distance is half of the link's capacity wasted?

> **Your answer:** *(write here)*

In [ ]:
delays = [1e-6, 1e-5, 1e-4, 3e-4, 1e-3, 1e-2, 1e-1, 1]          # one-way delay (s)
distance_km = [d * 2e8 / 1e3 for d in delays]
measured = [point_to_point(StopAndWaitSender, StopAndWaitReceiver, n_packets=50, payload_bytes=64,
                           rate_bps=1e6, prop_delay=d).utilisation for d in delays]
# Extension: add a "theory" curve for the utilisation and compare (ask Copilot to help).
sweep_plot(distance_km, {"simulated": measured}, "distance (km)", "fraction of link rate used",
           logx=True, title="Stop-and-wait over distance");

**C2.** At what distance is half the capacity wasted? What would the utilisation be for Mars, and what would you change in the protocol?

> **Your answer:** *(write here)*

### C3. How good is error detection, really?
Lecture 5 said parity is weak and a CRC is strong but not perfect. Count how many **corrupted payloads reach Bob's application** when stop-and-wait runs with each check (2000 payloads of 8 bytes).

**Predict:** rank `none`, `parity`, `crc8`, `crc16` at BER = 10⁻².

> **Your answer:** *(write here)*

In [ ]:
bers = [1e-3, 3e-3, 1e-2, 3e-2]
print(f"{'check':<8}" + "".join(f"BER {b:<9g}" for b in bers))
for check in ["none", "parity", "crc8", "crc16"]:
    row = []
    for ber in bers:
        r = point_to_point(StopAndWaitSender, StopAndWaitReceiver, n_packets=2000, payload_bytes=8,
                           rate_bps=1e6, prop_delay=1e-5, ber=ber, check=check)
        row.append(r.undetected_errors)
    print(f"{check:<8}" + "".join(f"{n:<13}" for n in row))

**C3.** Explain the parity row. Why does it get so bad at high BER, even though ARQ keeps retrying?

> **Your answer:** *(write here)*

## Part D: wrap up (2 min)

Your `student/mm5_link.py` stays in the framework. In **MM6** your link gets neighbours: several Alices share one channel to a base station, and their frames can collide.

After this session, a reference stop-and-wait is added to the framework (`commsim.reference`). MM6 uses it by default, so a bug in anyone's MM5 code cannot block the next lab. You can still plug in your own.

**D1.** One sentence: what surprised you most today?

> **Your answer:** *(write here)*